# Face Recognition Using Higher-Order Singular Value Decomposition (HOSVD)

This notebook implements face recognition using a third-order tensor representation and Higher-Order Singular Value Decomposition (HOSVD). Training images are organized by subject, and each test image is assigned to the subject with the smallest reconstruction error.

## 1. Import Libraries

Import the required libraries for image processing, tensor operations, numerical computation, and model evaluation.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import drive
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

drive.mount('/content/gdrive')

## 2. Load and Preprocess Training Images

The training dataset contains 15 subjects, with 9 images for each subject. Each image is converted to grayscale and resized to 270 × 288 pixels. The resulting images are flattened into vectors of length 77,760.

The training images are arranged in a third-order tensor with dimensions (pixels, images per subject, subjects).

In [ ]:
train_path = '/content/gdrive/My Drive/train_facedetection'

num_subjects = 15
images_per_subject = 9
image_height = 270
image_width = 288
num_pixels = image_height * image_width

A = np.zeros(
    (num_pixels, images_per_subject, num_subjects),
    dtype=np.float32
)

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

for subject_idx in range(num_subjects):
    subject_folder = os.path.join(train_path, f'sub{subject_idx + 1}')

    image_files = sorted([
        f for f in os.listdir(subject_folder)
        if f.lower().endswith(valid_extensions)
    ])

    if len(image_files) != images_per_subject:
        raise ValueError(
            f'Subject {subject_idx + 1} has {len(image_files)} images; '
            f'exactly {images_per_subject} images are required.'
        )

    for image_idx, filename in enumerate(image_files):
        image_path = os.path.join(subject_folder, filename)

        image = Image.open(image_path).convert('L')
        image = image.resize((image_width, image_height))

        image_array = np.asarray(image, dtype=np.float32)
        A[:, image_idx, subject_idx] = image_array.reshape(-1)

print('Training tensor shape:', A.shape)

## 3. Visualize Training Images

Display a sample image from each subject to verify that the images have been loaded and preprocessed correctly.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for subject_idx, ax in enumerate(axes.flat):
    image = A[:, 0, subject_idx].reshape(image_height, image_width)
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Subject {subject_idx + 1}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Tensor Unfolding

Unfolding converts a tensor into a matrix along a selected mode. For a third-order tensor, the three unfolding matrices correspond to modes 0, 1, and 2.

This operation is required to perform matrix multiplication and Singular Value Decomposition along each tensor mode.

In [ ]:
def unfolding(tensor, mode):
    tensor = np.asarray(tensor)

    if mode not in (0, 1, 2):
        raise ValueError('Mode must be 0, 1, or 2.')

    reordered = np.moveaxis(tensor, mode, 0)
    return reordered.reshape(tensor.shape[mode], -1)


for mode in range(3):
    print(f'Mode-{mode} unfolding shape:', unfolding(A, mode).shape)

## 5. Tensor Mode Product

The mode-n product multiplies a tensor by a matrix along a specified mode. The matrix dimensions must be compatible with the selected tensor dimension.

This operation is used to calculate the core tensor in HOSVD.

In [ ]:
def mode_product(tensor, matrix, mode):
    tensor = np.asarray(tensor)
    matrix = np.asarray(matrix)

    if matrix.shape[1] != tensor.shape[mode]:
        raise ValueError('Matrix dimensions are incompatible with the selected tensor mode.')

    result = np.tensordot(matrix, tensor, axes=(1, mode))
    return np.moveaxis(result, 0, mode)


# Example: multiply the training tensor along mode 0
example_result = mode_product(A, np.eye(A.shape[0]), 0)
print('Result shape:', example_result.shape)

## 6. Modal Singular Value Decomposition

HOSVD performs SVD on each mode-unfolded matrix. The left singular vectors of each unfolding form the factor matrices U1, U2, and U3.

Reduced SVD is used to avoid unnecessary computation and memory usage.

In [ ]:
U_list = []
singular_values = []

for mode in range(3):
    unfolded = unfolding(A, mode)
    U, S, VT = np.linalg.svd(unfolded, full_matrices=False)

    U_list.append(U)
    singular_values.append(S)

    print(f'U{mode + 1} shape:', U.shape)
    print(f'Mode-{mode} singular values shape:', S.shape)

## 7. Compute the Core Tensor

The core tensor is obtained by multiplying the original tensor by the transpose of each HOSVD factor matrix:

\[G = A \times_1 U_1^T \times_2 U_2^T \times_3 U_3^T\]

The core tensor stores the interactions between the latent components of the three modes.

In [ ]:
G = A.copy()

for mode in range(3):
    G = mode_product(G, U_list[mode].T, mode)

print('Core tensor shape:', G.shape)

## 8. Construct Subject-Specific Subspaces

Each frontal slice of the core tensor corresponds to one subject. For each subject, the corresponding slice is used to construct a low-dimensional subspace using QR decomposition.

The orthonormal columns of each Q matrix represent the subject-specific subspace used for recognition.

In [ ]:
subject_subspaces = []

for subject_idx in range(num_subjects):
    subject_matrix = G[:, :, subject_idx]

    Q, R = np.linalg.qr(subject_matrix, mode='reduced')
    subject_subspaces.append(Q)

    print(
        f'Subject {subject_idx + 1}: '
        f'subspace shape = {Q.shape}'
    )

## 9. Load and Preprocess Test Images

Load the test images and apply the same grayscale conversion and resizing procedure used for the training images. Each test image is represented as a vector with 77,760 elements.

In [ ]:
test_path = '/content/gdrive/My Drive/test_facedetection'

test_files = sorted([
    f for f in os.listdir(test_path)
    if f.lower().endswith(valid_extensions)
])

Z = []

for filename in test_files:
    image_path = os.path.join(test_path, filename)

    image = Image.open(image_path).convert('L')
    image = image.resize((image_width, image_height))

    image_array = np.asarray(image, dtype=np.float32)
    Z.append(image_array.reshape(-1))

Z = np.asarray(Z, dtype=np.float32)

print('Number of test images:', len(Z))
print('Test data shape:', Z.shape)

## 10. Face Recognition

Each test image is projected onto the first-mode HOSVD subspace:

\[z_{projected} = U_1^T z\]

For each subject, the reconstruction error is calculated by projecting the test vector onto that subject's orthonormal subspace:

\[e_p = \|z_{projected} - Q_p Q_p^T z_{projected}\|_2\]

The predicted identity is the subject with the smallest reconstruction error. This guarantees exactly one predicted label for each test image.

In [ ]:
y_pred = []
all_errors = []

U1 = U_list[0]

for test_vector in Z:
    projected_vector = U1.T @ test_vector

    subject_errors = []

    for Q in subject_subspaces:
        reconstructed = Q @ (Q.T @ projected_vector)
        error = np.linalg.norm(projected_vector - reconstructed)
        subject_errors.append(error)

    predicted_subject = int(np.argmin(subject_errors))

    y_pred.append(predicted_subject)
    all_errors.append(subject_errors)

y_pred = np.asarray(y_pred)
all_errors = np.asarray(all_errors)

print('Predicted subject labels:')
print(y_pred)
print('Prediction array shape:', y_pred.shape)

## 11. Evaluate Recognition Accuracy

The ground-truth labels are compared with the predicted subject labels. Labels use zero-based indexing, where 0 represents Subject 1 and 14 represents Subject 15.

The supplied ground-truth sequence is retained from the original implementation. Its order must correspond exactly to the sorted test image filenames.

In [ ]:
y_true = [
    0, 0, 1, 1, 2, 2, 3, 3,
    4, 4, 5, 5, 7, 7, 8, 8,
    9, 9, 10, 10, 11, 11, 10, 10,
    11, 11, 12, 12, 13, 13, 14, 14
]

if len(y_true) != len(y_pred):
    raise ValueError(
        f'The number of ground-truth labels ({len(y_true)}) '
        f'does not match the number of test images ({len(y_pred)}).'
    )

accuracy = accuracy_score(y_true, y_pred)

print(f'Face Recognition Accuracy: {accuracy:.4f}')
print(f'Face Recognition Accuracy: {accuracy * 100:.2f}%')

print('\nClassification Report:')
print(classification_report(y_true, y_pred, labels=list(range(15)), zero_division=0))

## 12. Confusion Matrix

The confusion matrix provides a detailed view of the classification results by comparing the true and predicted identities for all subjects.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_subjects)))

plt.figure(figsize=(11, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=[f'S{i + 1}' for i in range(num_subjects)],
    yticklabels=[f'S{i + 1}' for i in range(num_subjects)]
)

plt.xlabel('Predicted Subject')
plt.ylabel('True Subject')
plt.title('HOSVD Face Recognition Confusion Matrix')
plt.tight_layout()
plt.show()

## 13. Visualize Test Predictions

Display a selection of test images together with their predicted and actual subject identities. This provides a visual inspection of the recognition results.

In [ ]:
num_to_show = min(12, len(Z))
cols = 4
rows = int(np.ceil(num_to_show / cols))

fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
axes = np.asarray(axes).reshape(-1)

for idx in range(num_to_show):
    axes[idx].imshow(Z[idx].reshape(image_height, image_width), cmap='gray')
    axes[idx].set_title(
        f'True: S{y_true[idx] + 1}\nPredicted: S{y_pred[idx] + 1}'
    )
    axes[idx].axis('off')

for idx in range(num_to_show, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 14. Conclusion

In this experiment, face recognition was performed using Higher-Order Singular Value Decomposition. The training images were represented as a third-order tensor, and HOSVD was used to extract latent representations along the pixel, image, and subject modes. Subject-specific subspaces were constructed from the core tensor using QR decomposition. Test images were classified according to their minimum reconstruction error, and the results were evaluated using accuracy and a confusion matrix.